# Financial Statement Quality & Earnings Surprise Model

Academic Python research prototype by Vicente Pozo. Original development: August 2026; technical revision: 16 October 2026. Developed with AI assistance.

## Research question
How are accounting-quality measures and earnings surprises associated with subsequent benchmark-adjusted equity returns?

## Execution status
The revised notebook has passed format, syntax and targeted offline calculation checks. A complete real-data run has **not** been validated: SEC access needs a user-configured identifying User-Agent, and Yahoo returned HTTP 429 in this environment. No investment-performance conclusion is presented.

## Methods and assumptions
SEC filing-date availability, comparable trailing-twelve-month flows, trailing-history signal buckets and a purged chronological holdout. Entry is the first trading close strictly after the announcement date. The universe is a small, fixed convenience sample, with selection and survivorship limitations. Data-provider revisions can limit historical reproducibility. This is an association study, not evidence of tradable alpha.

In [ ]:
# Install requirements.txt in a dedicated environment before running.
# Colab: run %pip install -r requirements.txt after uploading the project files.
import os
import time
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11})
SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "")
# Set SEC_USER_AGENT to your own project name and contact address before live access.
# The contact value is sent to SEC, not committed to the repository.
SEC_HEADERS = {"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate"}
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_FACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
SEC_SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE_DIR = ROOT_DIR / "data/cache"
OUTPUT_DIR = ROOT_DIR / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UNIVERSE = ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "CAT"]
BENCHMARK = "^GSPC"
START_DATE = "2018-01-01"
END_DATE = "2026-10-16"  # fixed snapshot boundary; Yahoo end date is exclusive
MAX_FILINGS = 20
print(f"Configured: {len(UNIVERSE)} stocks, {START_DATE} to {END_DATE} (exclusive)")

## SEC access and caching
SEC requires identification for automated access. Set `SEC_USER_AGENT` in your environment to a project name and your chosen contact address. Do not commit personal contact settings. Requests stop on access denial or rate limits; cached responses reduce repeated downloads. Only the recent-submissions endpoint is used, so coverage is limited to its returned filings and `MAX_FILINGS`.

In [ ]:
def safe_get_json(url, headers=None, timeout=30):
    if not SEC_USER_AGENT or "@" not in SEC_USER_AGENT or "example.com" in SEC_USER_AGENT:
        raise RuntimeError("Configure SEC_USER_AGENT with your own project name and contact address before a live run. See README.md.")
    response = requests.get(url, headers=headers or SEC_HEADERS, timeout=timeout)
    if response.status_code in (403, 429):
        raise RuntimeError(f"Provider access/rate limit ({response.status_code}); stop and retry later under the provider policy.")
    response.raise_for_status()
    return response.json()

def cached_json(url, path, headers=None, max_age_hours=24):
    path = Path(path)
    if path.exists() and (time.time() - path.stat().st_mtime)/3600 <= max_age_hours:
        return json.loads(path.read_text(encoding="utf-8"))
    payload = safe_get_json(url, headers=headers)
    path.write_text(json.dumps(payload), encoding="utf-8")
    return payload


def numeric(values):
    return pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)

sec_tickers = cached_json(SEC_TICKERS_URL, CACHE_DIR/"sec_company_tickers.json", SEC_HEADERS, 72)
SEC_MAP = {str(v["ticker"]).upper(): {"cik": str(v["cik_str"]).zfill(10), "title": v["title"]}
           for v in sec_tickers.values() if v.get("ticker")}
print("SEC mappings:", sum(t in SEC_MAP for t in UNIVERSE))

## Market data
Adjusted daily closes are downloaded one ticker at a time. Failures are recorded and missing values remain missing. A failed universe download stops the notebook. The code does not change providers or routes to bypass restrictions.

In [ ]:
def download_close(tickers):
    frames, failures = {}, []
    for ticker in tickers:
        try:
            df = yf.download(
                ticker, start=START_DATE, end=END_DATE,
                auto_adjust=True, progress=False, threads=False, timeout=20
            )
            if df.empty:
                raise ValueError("No rows returned")
            close = df["Close"]
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            close = numeric(close).dropna()
            close.index = pd.to_datetime(close.index).tz_localize(None)
            frames[ticker] = close.rename(ticker)
            time.sleep(0.2)
        except Exception as exc:
            failures.append({"ticker": ticker, "error": str(exc)})
    return (pd.concat(frames.values(), axis=1).sort_index() if frames else pd.DataFrame()), pd.DataFrame(failures)

prices, price_failures = download_close(UNIVERSE + [BENCHMARK])
available = [t for t in UNIVERSE if t in prices.columns]

if prices.empty or not available:
    raise RuntimeError("Market data download failed.")

print(f"Downloaded {len(available)}/{len(UNIVERSE)} stocks.")
if not price_failures.empty:
    display(price_failures)

norm = prices[available].div(prices[available].iloc[0]).mul(100)
fig = px.line(norm, title="Normalized Equity Performance", labels={"value":"Indexed price","variable":"Ticker"})
fig.update_layout(template="plotly_white", height=500)
fig.show()

## Filing-time accounting inputs
Flows use comparable trailing-twelve-month (TTM) periods. For quarterly disclosures, TTM is the latest annual amount plus current fiscal YTD minus comparable prior fiscal YTD, using facts disclosed by that filing date. Missing compatible periods remain missing. Balance-sheet values are matched to period ends; average assets use the current and year-earlier balances. Alternative USD tags are combined, including instant facts without `start` dates.

In [ ]:
FACT_TAGS = {
    "revenue": [
        ("us-gaap","RevenueFromContractWithCustomerExcludingAssessedTax"),
        ("us-gaap","Revenues"),
        ("us-gaap","SalesRevenueNet"),
    ],
    "net_income": [("us-gaap","NetIncomeLoss")],
    "cfo": [("us-gaap","NetCashProvidedByUsedInOperatingActivities")],
    "gross_profit": [("us-gaap","GrossProfit")],
    "assets": [("us-gaap","Assets")],
    "liabilities": [("us-gaap","Liabilities")],
    "accounts_receivable": [
        ("us-gaap","AccountsReceivableNetCurrent"),
        ("us-gaap","AccountsReceivableCurrent"),
    ],
    "inventory": [("us-gaap","InventoryNet"), ("us-gaap","InventoryGross")],
}


def load_sec(ticker):
    if ticker not in SEC_MAP:
        raise KeyError(f"No CIK for {ticker}")
    cik = SEC_MAP[ticker]["cik"]
    facts = cached_json(
        SEC_FACTS_URL.format(cik=cik),
        CACHE_DIR / f"{ticker}_facts.json",
        headers=SEC_HEADERS,
        max_age_hours=24,
    )
    time.sleep(0.15)
    submissions = cached_json(
        SEC_SUBMISSIONS_URL.format(cik=cik),
        CACHE_DIR / f"{ticker}_submissions.json",
        headers=SEC_HEADERS,
        max_age_hours=24,
    )
    time.sleep(0.15)
    return facts, submissions

def fact_table(facts, name, fact_tags):
    """Combine alternative USD tags; retain filing dates and tag priority."""
    parts = []
    for priority, (taxonomy, tag) in enumerate(fact_tags[name]):
        units = facts.get("facts", {}).get(taxonomy, {}).get(tag, {}).get("units", {})
        records = units.get("USD", [])
        if not records:
            continue
        frame = pd.DataFrame(records)
        for column in ("start", "end", "filed"):
            if column not in frame:
                frame[column] = pd.NaT
            frame[column] = pd.to_datetime(frame[column], errors="coerce")
        frame["value"] = numeric(frame["val"])
        frame["tag_priority"] = priority
        parts.append(frame.dropna(subset=["end", "filed", "value"]))
    if not parts:
        return pd.DataFrame(columns=["start", "end", "filed", "value", "tag_priority"])
    return pd.concat(parts, ignore_index=True).sort_values(["end", "filed", "tag_priority"])

def _latest_fact(frame):
    if frame.empty:
        return np.nan
    # Prefer latest disclosed revision, then the preferred accounting tag.
    return float(frame.sort_values(["filed", "tag_priority"], ascending=[False, True]).iloc[0]["value"])

def choose_ttm_flow(frame, filed_date, report_date):
    """Annual flow or FY + current YTD - comparable prior YTD, all known as of filing.

    Missing compatible periods remain NaN rather than mixing annual/quarterly
    amounts or treating missing cash flow as zero.
    """
    if frame.empty:
        return np.nan
    fd, rd = pd.Timestamp(filed_date), pd.Timestamp(report_date)
    x = frame[(frame["filed"] <= fd) & (frame["end"] <= rd + pd.Timedelta(days=7))].copy()
    duration = (x["end"] - x["start"]).dt.days
    annual = x[duration.between(300, 430)]
    direct = annual[(annual["end"] - rd).abs() <= pd.Timedelta(days=7)]
    if not direct.empty:
        return _latest_fact(direct)
    annual = annual[annual["end"] < rd - pd.Timedelta(days=7)]
    if annual.empty:
        return np.nan
    annual_end = annual["end"].max()
    base = annual[annual["end"] == annual_end].sort_values(
        ["filed", "tag_priority"], ascending=[False, True]).iloc[0]
    if (rd - annual_end).days > 310:
        return np.nan
    current = x[((x["end"] - rd).abs() <= pd.Timedelta(days=7)) &
                ((x["start"] - (annual_end + pd.Timedelta(days=1))).abs() <= pd.Timedelta(days=14))]
    prior_end = rd - pd.DateOffset(years=1)
    prior = x[((x["end"] - prior_end).abs() <= pd.Timedelta(days=14)) &
              ((x["start"] - base["start"]).abs() <= pd.Timedelta(days=14))]
    current_value, prior_value = _latest_fact(current), _latest_fact(prior)
    if not np.isfinite(current_value) or not np.isfinite(prior_value):
        return np.nan
    return float(base["value"] + current_value - prior_value)

def choose_balance(frame, filed_date, report_date):
    if frame.empty:
        return np.nan
    fd, rd = pd.Timestamp(filed_date), pd.Timestamp(report_date)
    x = frame[(frame["filed"] <= fd) &
              ((frame["end"] - rd).abs() <= pd.Timedelta(days=7))]
    return _latest_fact(x)

def build_filings(ticker):
    facts, submissions = load_sec(ticker)
    recent = pd.DataFrame(submissions.get("filings", {}).get("recent", {}))
    if recent.empty:
        return pd.DataFrame()
    for column in ("filingDate", "reportDate"):
        recent[column] = pd.to_datetime(recent[column], errors="coerce")
    recent = recent[recent["form"].isin(["10-Q", "10-K"])].dropna(subset=["filingDate", "reportDate"])
    recent = recent.sort_values("filingDate").tail(MAX_FILINGS)
    tables = {name: fact_table(facts, name, FACT_TAGS) for name in FACT_TAGS}
    rows = []
    for _, filing in recent.iterrows():
        fd, rd = filing["filingDate"], filing["reportDate"]
        prior = rd - pd.DateOffset(years=1)
        row = {"ticker": ticker, "company": SEC_MAP[ticker]["title"], "filing_date": fd,
               "report_date": rd, "form": filing["form"], "accession_number": filing.get("accessionNumber")}
        for name in ("revenue", "net_income", "cfo", "gross_profit"):
            row[name] = choose_ttm_flow(tables[name], fd, rd)
        for name in ("assets", "liabilities", "accounts_receivable", "inventory"):
            row[name] = choose_balance(tables[name], fd, rd)
        row["prior_year_assets"] = choose_balance(tables["assets"], fd, prior)
        row["prior_year_revenue"] = choose_ttm_flow(tables["revenue"], fd, prior)
        prior_gp = choose_ttm_flow(tables["gross_profit"], fd, prior)
        row["prior_year_gross_margin"] = prior_gp / row["prior_year_revenue"] if row["prior_year_revenue"] else np.nan
        rows.append(row)
    return pd.DataFrame(rows).drop_duplicates(["ticker", "filing_date"], keep="last")

sec_parts, sec_errors = [], []
for ticker in available:
    try:
        panel = build_filings(ticker)
        if not panel.empty:
            sec_parts.append(panel)
    except Exception as exc:
        sec_errors.append({"ticker": ticker, "error": str(exc)})
sec = pd.concat(sec_parts, ignore_index=True) if sec_parts else pd.DataFrame()
if sec_errors:
    display(pd.DataFrame(sec_errors))
if sec.empty:
    raise RuntimeError("No SEC filing panel available; inspect source access and coverage.")
print("Filing rows:", len(sec))

## Accounting-quality signal
At each filing date, rank available metrics against the latest observation of each issuer known at that date. Scores are normalized by the absolute weights of available features and require at least 50% feature-weight coverage. There is no whole-sample winsorization or ranking against later filings. Because the universe is small and contains financial and non-financial companies, industry comparability remains a material limitation.

In [ ]:
WEIGHTS = {"cfo_ni_ratio": 1., "accrual_ratio": -1., "roaa": 1.,
           "gross_margin_change": .75, "revenue_growth": .5, "leverage": -.75}

def compute_quality(features, min_peers=5):
    """Score each filing using a one-row-per-issuer information set available then."""
    out = features.sort_values(["filing_date", "ticker"]).reset_index(drop=True).copy()
    out["gross_margin"] = out["gross_profit"] / out["revenue"].replace(0, np.nan)
    out["revenue_growth"] = out["revenue"] / out["prior_year_revenue"].replace(0, np.nan) - 1
    out["gross_margin_change"] = out["gross_margin"] - out["prior_year_gross_margin"]
    out["avg_assets"] = (out["assets"] + out["prior_year_assets"]) / 2
    denom = out["avg_assets"].where(out["avg_assets"] > 0)
    out["roaa"] = out["net_income"] / denom
    out["accrual_ratio"] = (out["net_income"] - out["cfo"]) / denom
    out["cfo_ni_ratio"] = out["cfo"] / out["net_income"].where(out["net_income"] > 0)
    out["leverage"] = out["liabilities"] / out["assets"].where(out["assets"] > 0)
    for metric in WEIGHTS:
        out[metric] = numeric(out[metric])
    out["quality_score"] = np.nan
    out["quality_coverage"] = 0.
    out["quality_peer_count"] = 0
    for day, indices in out.groupby("filing_date", sort=True).groups.items():
        known = out.loc[out["filing_date"] <= day].drop_duplicates("ticker", keep="last").set_index("ticker")
        scores, coverages = [], []
        for metric, weight in WEIGHTS.items():
            values = known[metric]
            count = values.notna().sum()
            rank = values.rank(method="average")
            centered = (2 * (rank - 1) / (count - 1) - 1) if count >= min_peers else values * np.nan
            scores.append(centered * weight)
            coverages.append(centered.notna() * abs(weight))
        numerator = pd.concat(scores, axis=1).sum(axis=1, min_count=1)
        denominator = pd.concat(coverages, axis=1).sum(axis=1)
        score = (numerator / denominator.replace(0, np.nan)).where(
            denominator / sum(abs(w) for w in WEIGHTS.values()) >= .5)
        for index in indices:
            ticker = out.at[index, "ticker"]
            out.at[index, "quality_score"] = score.get(ticker, np.nan)
            out.at[index, "quality_coverage"] = denominator.get(ticker, 0) / sum(abs(w) for w in WEIGHTS.values())
            out.at[index, "quality_peer_count"] = len(known)
    return out

sec = compute_quality(sec)
display(sec.groupby("ticker").agg(filings=("filing_date", "count"),
    median_quality=("quality_score", "median"), coverage=("quality_coverage", "median")))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=sec.dropna(subset=["quality_score"]), x="ticker", y="quality_score", ax=axes[0])
axes[0].set_title("Accounting quality by filing-date information set")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y", alpha=0.25)

corr_cols = ["quality_score","cfo_ni_ratio","accrual_ratio","roaa","gross_margin_change","revenue_growth","leverage"]
sns.heatmap(sec[corr_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1])
axes[1].set_title("Accounting Signal Correlation")
plt.tight_layout()
plt.show()

## Earnings announcements
EPS surprise is computed as `100 × (reported EPS − estimated EPS) / abs(estimated EPS)`. Zero or missing estimates remain missing. This avoids guessing whether a vendor surprise field is a fraction or percentage. Announcement timestamps are normalized to New York dates; the conservative next-close entry rule does not depend on precise intraday execution.

In [ ]:
def earnings_history(ticker, limit=40):
    frame = yf.Ticker(ticker).get_earnings_dates(limit=limit)
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame = frame.reset_index()
    frame = frame.rename(columns={frame.columns[0]: "earnings_datetime", "EPS Estimate": "eps_estimate", "Reported EPS": "reported_eps"})
    def local_timestamp(value):
        value = pd.Timestamp(value)
        if pd.isna(value):
            return pd.NaT
        return value.tz_localize("America/New_York") if value.tzinfo is None else value.tz_convert("America/New_York")
    frame["earnings_datetime"] = frame["earnings_datetime"].map(local_timestamp)
    frame["earnings_date"] = frame["earnings_datetime"].map(lambda v: v.tz_localize(None).normalize() if pd.notna(v) else pd.NaT)
    for name in ("eps_estimate", "reported_eps"):
        frame[name] = numeric(frame[name]) if name in frame else np.nan
    denominator = frame["eps_estimate"].abs().replace(0, np.nan)
    frame["surprise_pct"] = 100 * (frame["reported_eps"] - frame["eps_estimate"]) / denominator
    frame["ticker"] = ticker
    return frame[["ticker", "earnings_datetime", "earnings_date", "eps_estimate", "reported_eps", "surprise_pct"]]

earn_parts, earnings_errors = [], []
for ticker in available:
    try:
        part = earnings_history(ticker)
        if not part.empty:
            earn_parts.append(part)
    except Exception as exc:
        earnings_errors.append({"ticker": ticker, "error": str(exc)})
    time.sleep(.2)
if earnings_errors:
    display(pd.DataFrame(earnings_errors))
if not earn_parts:
    raise RuntimeError("No earnings history returned by the data provider.")
earnings = pd.concat(earn_parts, ignore_index=True).drop_duplicates(["ticker", "earnings_datetime"])
earnings = earnings[earnings["earnings_date"].between(pd.Timestamp(START_DATE), pd.Timestamp(END_DATE) - pd.Timedelta(days=1))]
earnings = earnings.dropna(subset=["surprise_pct"]).copy()
if earnings.empty:
    raise RuntimeError("No events with numeric EPS surprises in the selected period.")
display(earnings.head(10))
fig = px.histogram(earnings, x="surprise_pct", nbins=60, title="Observed EPS surprises", labels={"surprise_pct": "EPS surprise (%)"})
fig.update_layout(template="plotly_white", height=450)
fig.show()

## Event information set and composite
Match each announcement to a filing strictly before its New York calendar date. Sort globally by the date merge key, then ticker. The composite equally weights accounting quality and `tanh(EPS surprise / 20)`. The 20-percentage-point scale is an explicit, unoptimized modeling choice, not an estimated optimal weight. No same-day sparse cross-sectional z-scores are used.

In [ ]:
def match_events(earnings, features):
    left = earnings.dropna(subset=["earnings_date"]).sort_values(["earnings_date", "ticker"])
    right = features.dropna(subset=["filing_date"]).sort_values(["filing_date", "ticker"])
    events = pd.merge_asof(left, right, left_on="earnings_date", right_on="filing_date",
                           by="ticker", direction="backward", allow_exact_matches=False)
    events["filing_precedes_earnings"] = events["filing_date"] < events["earnings_date"]
    events = events.dropna(subset=["quality_score", "surprise_pct"]).copy()
    # Fixed ex-ante 20-percentage-point scale avoids same-day sparse z-scores.
    events["surprise_signal"] = np.tanh(events["surprise_pct"] / 20.)
    events["composite_score"] = .5 * events["surprise_signal"] + .5 * events["quality_score"]
    if not events["filing_precedes_earnings"].all():
        raise AssertionError("A filing did not precede its matched announcement.")
    return events.sort_values(["earnings_date", "ticker"]).reset_index(drop=True)

events = match_events(earnings, sec)
if events.empty:
    raise RuntimeError("No events have both eligible filing features and EPS surprises.")
display(events[["ticker", "earnings_date", "filing_date", "quality_score", "surprise_pct", "composite_score"]].head(12))

## Forward returns and event studies
Entry is the first benchmark trading close strictly after the announcement date. A horizon of `h` sessions uses `h+1` prices: entry and the close `h` observations later. Holding-period benchmark-adjusted returns subtract benchmark buy-and-hold returns. Missing or nonpositive entry/exit prices remain unavailable. These are gross observational returns, without transaction costs or an implemented portfolio.

In [ ]:
def attach_forward_returns(events, prices, benchmark="^GSPC", horizons=(1, 5, 21)):
    out = events.copy()
    if benchmark not in prices:
        raise ValueError("Missing benchmark prices.")
    index = prices[benchmark].dropna().index.sort_values()
    for horizon in horizons:
        entries, exits, stock_returns, adjusted_returns = [], [], [], []
        for _, event in out.iterrows():
            dates = index[index > pd.Timestamp(event["earnings_date"])]
            entry = exit_date = pd.NaT
            stock_return = adjusted = np.nan
            if event["ticker"] in prices and len(dates) > horizon:
                entry, exit_date = dates[0], dates[horizon]
                s0, s1 = prices.loc[[entry, exit_date], event["ticker"]]
                b0, b1 = prices.loc[[entry, exit_date], benchmark]
                if np.isfinite([s0, s1, b0, b1]).all() and min(s0, s1, b0, b1) > 0:
                    stock_return = s1 / s0 - 1
                    adjusted = stock_return - (b1 / b0 - 1)
            entries.append(entry); exits.append(exit_date)
            stock_returns.append(stock_return); adjusted_returns.append(adjusted)
        out[f"entry_{horizon}d"] = entries
        out[f"exit_{horizon}d"] = exits
        out[f"return_{horizon}d"] = stock_returns
        out[f"abnormal_return_{horizon}d"] = adjusted_returns
    return out

events = attach_forward_returns(events, prices, BENCHMARK)
display(events[["ticker", "earnings_date", "entry_1d", "exit_1d", "return_1d", "abnormal_return_5d", "abnormal_return_21d"]].head(10))

## Trailing-history buckets
For each event, construct five signal buckets from cutoffs in the preceding 365 calendar days, excluding the current announcement date. At least 20 historical events and distinct thresholds are required. Sparse samples remain unclassified. These are historical-reference buckets, not a simultaneous tradable long-short portfolio. Naive t-statistics below are descriptive because overlapping returns and repeated issuers are dependent.

In [ ]:
def assign_history_buckets(events, value_column, lookback_days=365, min_history=20):
    """Use only observations strictly before each announcement date for cutoffs."""
    out = pd.Series(np.nan, index=events.index)
    for day, indices in events.groupby("earnings_date", sort=True).groups.items():
        history = events.loc[(events["earnings_date"] < day) &
                             (events["earnings_date"] >= day - pd.Timedelta(days=lookback_days)), value_column].dropna()
        if len(history) < min_history or history.nunique() < 5:
            continue
        thresholds = history.quantile([.2, .4, .6, .8]).to_numpy()
        if len(np.unique(thresholds)) < 4:
            continue
        current = events.loc[indices, value_column].dropna()
        out.loc[current.index] = np.searchsorted(thresholds, current.to_numpy(), side="right") + 1
    return out

for value, bucket in [("surprise_pct", "surprise_quintile"), ("quality_score", "quality_quintile"), ("composite_score", "composite_quintile")]:
    events[bucket] = assign_history_buckets(events, value)

def bucket_table(bucket_col):
    t = events.dropna(subset=[bucket_col,"abnormal_return_21d"]).groupby(bucket_col)["abnormal_return_21d"].agg(
        count="count", mean="mean", median="median", std="std"
    )
    t["t_stat"] = t["mean"] / (t["std"] / np.sqrt(t["count"]))
    return t

surprise_tbl = bucket_table("surprise_quintile")
quality_tbl = bucket_table("quality_quintile")
composite_tbl = bucket_table("composite_quintile")

print("Surprise quintiles")
display(surprise_tbl.style.format("{:.4f}"))
print("Quality quintiles")
display(quality_tbl.style.format("{:.4f}"))
print("Composite quintiles")
display(composite_tbl.style.format("{:.4f}"))

comparison = pd.DataFrame({
    "signal":["Earnings Surprise","Accounting Quality","Composite"],
    "Q5-Q1 abnormal return":[
        surprise_tbl.loc[5,"mean"] - surprise_tbl.loc[1,"mean"] if {1,5}.issubset(surprise_tbl.index) else np.nan,
        quality_tbl.loc[5,"mean"] - quality_tbl.loc[1,"mean"] if {1,5}.issubset(quality_tbl.index) else np.nan,
        composite_tbl.loc[5,"mean"] - composite_tbl.loc[1,"mean"] if {1,5}.issubset(composite_tbl.index) else np.nan,
    ]
})
display(comparison.style.format({"Q5-Q1 abnormal return":"{:.2%}"}))

fig = px.bar(
    comparison, x="signal", y="Q5-Q1 abnormal return",
    title="Historical bucket spread: 21-session benchmark-adjusted return",
    labels={"Q5-Q1 abnormal return":"Return spread"}
)
fig.update_yaxes(tickformat=".1%")
fig.update_layout(template="plotly_white", height=450)
fig.show()

## Chronological holdout with label-maturity purge
Split by unique entry dates, holding the latest 25% of entry dates for testing. Remove training events whose 21-session exit is on or after the test cutoff. Fit preprocessing and models only on the remaining training observations. Report test RMSE against a training-mean baseline. OLS is exploratory with HC3 errors; event/issuer dependence and small-sample uncertainty remain limitations.

In [ ]:
def chronological_holdout(events, target="abnormal_return_21d", fraction=.75):
    data = events.dropna(subset=[target, "entry_21d", "exit_21d"]).sort_values(["entry_21d", "ticker"])
    dates = data["entry_21d"].drop_duplicates().sort_values()
    if len(dates) < 4:
        return data.iloc[:0], data.iloc[:0]
    cutoff = dates.iloc[min(max(1, int(len(dates) * fraction)), len(dates) - 1)]
    train = data[(data["entry_21d"] < cutoff) & (data["exit_21d"] < cutoff)]
    test = data[data["entry_21d"] >= cutoff]
    return train, test

model_features = ["surprise_signal", "quality_score", "accrual_ratio", "roaa", "gross_margin_change", "revenue_growth", "leverage"]
train, test = chronological_holdout(events)
if len(train) >= 40 and len(test) >= 10:
    X_train = train[model_features].replace([np.inf, -np.inf], np.nan)
    X_test = test[model_features].replace([np.inf, -np.inf], np.nan)
    usable = X_train.columns[X_train.notna().any()]
    X_train, X_test = X_train[usable], X_test[usable]
    y_train, y_test = train["abnormal_return_21d"], test["abnormal_return_21d"]
    regression_imputer = SimpleImputer(strategy="median")
    train_filled = regression_imputer.fit_transform(X_train)
    ols = sm.OLS(y_train.to_numpy(), sm.add_constant(train_filled)).fit(cov_type="HC3")
    print(ols.summary())
    forest = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(
        n_estimators=400, max_depth=4, min_samples_leaf=8, random_state=RANDOM_SEED, n_jobs=-1))
    forest.fit(X_train, y_train)
    predicted = forest.predict(X_test)
    rmse = float(np.sqrt(np.mean((predicted-y_test)**2)))
    baseline_rmse = float(np.sqrt(np.mean((float(y_train.mean())-y_test)**2)))
    display(pd.DataFrame({"model": ["Random Forest", "Training-mean baseline"], "test_RMSE": [rmse, baseline_rmse]}))
    print("Training labels mature before first test entry:", train["exit_21d"].max() < test["entry_21d"].min())
    importance = pd.Series(forest[-1].feature_importances_, index=usable).sort_values()
    importance.plot.barh(title="Random Forest training feature importance", xlabel="Importance")
    plt.tight_layout(); plt.show()
else:
    print(f"Model evaluation skipped: {len(train)} training and {len(test)} test events after date split/purge.")

## FRED context and additive event-time CAR
FRED series are optional descriptive context using the latest available revisions, not point-in-time model inputs. The event profile sums daily stock-minus-benchmark returns from entry close over 1–21 sessions. It does not compound abnormal returns, and its definition differs from the holding-period adjusted return used above.

In [ ]:
def fred_csv(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={START_DATE}&coed={END_DATE}"
    try:
        frame = pd.read_csv(url)
        frame["observation_date"] = pd.to_datetime(frame["observation_date"])
        frame[series_id] = numeric(frame[series_id])
        return frame.set_index("observation_date")[[series_id]]
    except Exception as exc:
        print(f"Optional FRED series {series_id} unavailable: {exc}")
        return pd.DataFrame()

fred = pd.concat([fred_csv("DFF"), fred_csv("UNRATE")], axis=1)
if "DFF" in fred and fred["DFF"].notna().any():
    fred["DFF"].plot(title="Federal funds rate: latest-vintage context", ylabel="Percent")
    plt.tight_layout(); plt.show()


def event_profile(events, prices, bucket_value, benchmark="^GSPC", max_horizon=21):
    """Additive daily abnormal returns from entry close; includes h+1 prices."""
    sub = events[events["composite_quintile"] == bucket_value]
    index = prices[benchmark].dropna().index.sort_values()
    paths = []
    for _, event in sub.iterrows():
        dates = index[index > event["earnings_date"]][:max_horizon + 1]
        if len(dates) != max_horizon + 1:
            continue
        pair = prices.loc[dates, [event["ticker"], benchmark]]
        if pair.isna().any().any() or (pair <= 0).any().any():
            continue
        returns = pair.pct_change(fill_method=None).iloc[1:]
        paths.append((returns.iloc[:, 0] - returns.iloc[:, 1]).cumsum().to_numpy())
    return (pd.Series(np.vstack(paths).mean(axis=0), index=np.arange(1, max_horizon + 1))
            if paths else pd.Series(dtype=float))

fig, ax = plt.subplots()
for bucket, label in [(1, "Low composite"), (5, "High composite")]:
    profile = event_profile(events, prices, bucket, BENCHMARK)
    if not profile.empty:
        ax.plot(profile.index, profile*100, label=label)
ax.set(title="Additive daily CAR after entry close", xlabel="Trading sessions after entry", ylabel="CAR (%)")
if ax.lines:
    ax.legend()
plt.tight_layout(); plt.show()

## Research outputs and provenance
CSV/HTML exports are generated only after successful upstream execution. Retain the execution date, universe, parameters, row coverage and source configuration. Source caches and raw vendor data are excluded from the repository by default. Check provider terms before redistribution.

In [ ]:
issuer_summary = (
    sec.groupby("ticker")
    .agg(
        filings=("filing_date","count"),
        median_quality=("quality_score","median"),
        median_accrual=("accrual_ratio","median"),
        median_cfo_ni=("cfo_ni_ratio","median"),
        median_roaa=("roaa","median"),
    )
    .sort_values("median_quality", ascending=False)
)

score_rows = []
for signal, col in [
    ("Earnings Surprise","surprise_quintile"),
    ("Accounting Quality","quality_quintile"),
    ("Composite","composite_quintile"),
]:
    high = events.loc[events[col]==5,"abnormal_return_21d"].dropna()
    low = events.loc[events[col]==1,"abnormal_return_21d"].dropna()
    spread = high.mean()-low.mean() if len(high) and len(low) else np.nan
    t, p = stats.ttest_1samp(high, 0) if len(high) > 1 else (np.nan,np.nan)
    score_rows.append({
        "signal":signal,
        "high_n":len(high),
        "low_n":len(low),
        "high_mean_abnormal_21d":high.mean(),
        "low_mean_abnormal_21d":low.mean(),
        "q5_minus_q1":spread,
        "high_t_stat":t,
        "high_p_value":p,
    })
scorecard = pd.DataFrame(score_rows)

sec.to_csv(OUTPUT_DIR / "sec_filing_quality_panel.csv", index=False)
earnings.to_csv(OUTPUT_DIR / "earnings_events_raw.csv", index=False)
events.to_csv(OUTPUT_DIR / "earnings_quality_event_panel.csv", index=False)
issuer_summary.to_csv(OUTPUT_DIR / "issuer_quality_summary.csv")
scorecard.to_csv(OUTPUT_DIR / "research_scorecard.csv", index=False)

html_table = scorecard.to_html(index=False, float_format=lambda x: f"{x:.4f}")
report = """<!doctype html>
<html><head><meta charset="utf-8"><title>Quantamental Research Scorecard</title>
<style>body{font-family:Arial;margin:40px}table{border-collapse:collapse}th,td{border:1px solid #ddd;padding:7px}th{background:#f2f2f2}</style>
</head><body><h1>Financial Statement Quality & Earnings Surprise Model</h1>
<p>Generated automatically from the Colab notebook.</p>
<h2>Scorecard</h2>
""" + html_table + "</body></html>"
(OUTPUT_DIR / "research_scorecard.html").write_text(report, encoding="utf-8")

print("Saved:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

metadata = {"execution_date_utc": pd.Timestamp.now(tz="UTC").isoformat(),
            "universe": UNIVERSE, "start": START_DATE, "end_exclusive": END_DATE,
            "benchmark": BENCHMARK, "sec_rows": len(sec), "earnings_rows": len(earnings),
            "matched_events": len(events), "entry_rule": "first benchmark close strictly after earnings date",
            "source_urls": [SEC_TICKERS_URL, "https://data.sec.gov/", "https://finance.yahoo.com/", "https://fred.stlouisfed.org/"]}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata, indent=2))

## Interpretation and next steps
Interpret only the outputs of a successful run. No empirical result is embedded in this revised delivery. Do not treat a bucket spread or low RMSE as investment performance. Before external performance claims, assess data revisions, fiscal comparability, sector effects, overlapping labels, clustered uncertainty, transaction costs, universe construction and out-of-sample stability.

The notebook is an academic implementation and methodology demonstration. The repository contains targeted calculation checks and a change log. A complete live execution and rendered empirical outputs remain necessary before presenting validated research findings.